[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tuesdaythe13th/multilingualcompositionalsafety_evals/blob/main/safety_routing_colab.ipynb)

# ARTIFEX — Ethical AI Feedback Loop Analysis
### `safety_routing_colab.ipynb`

```
  ██████╗ ██████╗ ██╗   ██╗████████╗██╗███╗   ██╗ ██████╗
 ██╔══██╗██╔═══██╗██║   ██║╚══██╔══╝██║████╗  ██║██╔════╝
 ██████╔╝██║   ██║██║   ██║   ██║   ██║██╔██╗ ██║██║  ███╗
 ██╔══██╗██║   ██║██║   ██║   ██║   ██║██║╚██╗██║██║   ██║
 ██║  ██║╚██████╔╝╚██████╔╝   ██║   ██║██║ ╚████║╚██████╔╝
 ╚═╝  ╚═╝ ╚═════╝  ╚═════╝    ╚═╝   ╚═╝╚═╝  ╚═══╝ ╚═════╝
```

**Goal**: Cluster user feedback using Scikit-learn K-Means and generate per-cluster
natural-language summaries via OpenAI or Anthropic LLM APIs — with graceful
extractive fallback when no API keys are present.

**Pipeline overview**:
```
  Data Load → EDA → Embeddings → K-Means → LLM Summarize → Report
```

**Principal Engineer**: Tuesday · ARTIFEX Labs  
tuesday@artifex.fun · linktr.ee/artifexlabs · huggingface.co/222tuesday  
contact: zcal.co/tuesday

## Library Reference

| Library | Purpose |
|---|---|
| `scikit-learn` | K-Means clustering, PCA dimensionality reduction, silhouette scoring |
| `transformers` | Pretrained transformer model for text embedding generation |
| `datasets` | HuggingFace dataset hub (imdb, amazon_polarity fallback loaders) |
| `openai` / `anthropic` | LLM API clients for per-cluster summarization |
| `pandera` | Runtime DataFrame schema validation at ingestion |
| `ydata-profiling` | Automated EDA — null analysis, outliers, text length distributions |
| `bokeh` | Interactive scatter plot dashboard (cluster size vs. sentiment quality) |
| `matplotlib` / `seaborn` | Static cluster PCA projection charts |
| `watermark` | Reproducibility: records all package versions in Cell 11 |

In [ ]:
#@title ② ARTIFEX Header — Branded Display
from IPython.display import display, HTML
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

display(HTML(f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Syne+Mono&display=swap');
  .artifex-header {{
    font-family: 'Syne Mono', monospace;
    background: #0a0a0a;
    color: #e8ff00;
    padding: 24px 32px;
    border: 2px solid #e8ff00;
    border-radius: 4px;
    white-space: pre;
    line-height: 1.4;
  }}
  .artifex-sub {{ color: #aaaaaa; font-size: 0.85em; }}
</style>
<div class='artifex-header'>
 ██████╗ ██████╗ ██╗   ██╗████████╗██╗███╗   ██╗ ██████╗
 ██╔══██╗██╔═══██╗██║   ██║╚══██╔══╝██║████╗  ██║██╔════╝
 ██████╔╝██║   ██║██║   ██║   ██║   ██║██╔██╗ ██║██║  ███╗
 ██╔══██╗██║   ██║██║   ██║   ██║   ██║██║╚██╗██║██║   ██║
 ██║  ██║╚██████╔╝╚██████╔╝   ██║   ██║██║ ╚████║╚██████╔╝
 ╚═╝  ╚═╝ ╚═════╝  ╚═════╝    ╚═╝   ╚═╝╚═╝  ╚═══╝ ╚═════╝

 🔄 ETHICAL AI FEEDBACK LOOP ANALYSIS
 <span class='artifex-sub'>Timestamp: {ts}
 Principal Engineer: Tuesday · ARTIFEX Labs · tuesday@artifex.fun</span>
</div>
"""))

In [ ]:
#@title ① Install Dependencies
import subprocess, sys

PACKAGES = [
    'scikit-learn',
    'transformers',
    'datasets',
    'openai',
    'anthropic',
    'pandera',
    'ydata-profiling',
    'bokeh',
    'watermark',
    'torch',
    'sentence-transformers',
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ All dependencies installed.')

In [ ]:
#@title ③ Secure Secrets Ingestion
# Priority: Colab Secrets → Google Drive → file upload → env → None
import os

def _get_secret(name: str) -> str | None:
    # 1. Colab userdata
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # 2. Environment variable
    return os.environ.get(name)

OPENAI_API_KEY    = _get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = _get_secret('ANTHROPIC_API_KEY')
KAGGLE_USERNAME   = _get_secret('KAGGLE_USERNAME')
KAGGLE_KEY        = _get_secret('KAGGLE_KEY')

print('OPENAI_API_KEY:    ', '✅ present' if OPENAI_API_KEY    else '⚠️  not found (extractive fallback active)')
print('ANTHROPIC_API_KEY: ', '✅ present' if ANTHROPIC_API_KEY else '⚠️  not found (extractive fallback active)')
print('KAGGLE creds:      ', '✅ present' if KAGGLE_USERNAME   else '⚠️  not found (Kaggle fallback disabled)')

In [ ]:
#@title ④ Data Loading — Multi-Source with Unified Schema
import pandas as pd
import pandera as pa
from pandera import Column, DataFrameSchema, Check
import hashlib

MAX_ROWS = 2000

FEEDBACK_SCHEMA = DataFrameSchema({
    'text':      Column(str, Check(lambda s: s.str.len() > 0)),
    'label':     Column(str, nullable=True),
    'source':    Column(str, nullable=True),
})

def _load_csv_upload() -> pd.DataFrame | None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            name = list(uploaded.keys())[0]
            df = pd.read_csv(name)
            if 'text' in df.columns:
                df['source'] = 'upload'
                return df
    except Exception:
        pass
    return None

def _load_hf(dataset_name: str, split: str = 'train', text_col: str = 'text', label_col: str = 'label') -> pd.DataFrame:
    from datasets import load_dataset
    ds = load_dataset(dataset_name, split=f'{split}[:{MAX_ROWS}]')
    df = ds.to_pandas()[[text_col]].rename(columns={text_col: 'text'})
    if label_col in ds.features:
        df['label'] = ds.to_pandas()[label_col].astype(str)
    else:
        df['label'] = None
    df['source'] = dataset_name
    return df

# Try CSV upload first, else HuggingFace
df_raw = _load_csv_upload()

if df_raw is None:
    print('No CSV upload. Loading from HuggingFace (imdb)...')
    try:
        df_raw = _load_hf('imdb', text_col='text', label_col='label')
    except Exception as e:
        print(f'imdb failed ({e}). Trying amazon_polarity...')
        df_raw = _load_hf('amazon_polarity', text_col='content', label_col='label')

# Unify schema
for col in ['text', 'label', 'source']:
    if col not in df_raw.columns:
        df_raw[col] = None

df_raw = df_raw[['text', 'label', 'source']].copy()
df_raw['text'] = df_raw['text'].astype(str).str.strip()

# Deduplicate via hash
df_raw['_hash'] = df_raw['text'].apply(lambda t: hashlib.md5(t.encode()).hexdigest())
df_raw = df_raw.drop_duplicates(subset='_hash').drop(columns='_hash').reset_index(drop=True)
df_raw = df_raw.head(MAX_ROWS)

# Validate schema
df = FEEDBACK_SCHEMA.validate(df_raw)
print(f'✅ Loaded {len(df)} rows from [{df.source.unique().tolist()}]')
display(df.head(3))

In [ ]:
#@title ⑤ Pipeline Graph + Automated EDA
from IPython.display import display, HTML
import matplotlib.pyplot as plt

# Pipeline ASCII graph
display(HTML('<pre style="font-family:monospace;background:#111;color:#e8ff00;padding:16px;">\n'
             '  ┌──────────────────────────────────────────────────────────────────┐\n'
             '  │               FEEDBACK LOOP PIPELINE                            │\n'
             '  ├──────────┬────────────┬─────────────┬───────────┬───────────────┤\n'
             '  │  LOAD    │  EMBED     │  K-MEANS    │  LLM SUM  │  REPORT       │\n'
             '  │  CSV/HF  │  sentence- │  k=5-10     │  OpenAI/  │  HTML +       │\n'
             '  │  Kaggle  │  transform │  PCA 2D     │  Anthropic│  Bokeh dash   │\n'
             '  └──────────┴────────────┴─────────────┴───────────┴───────────────┘\n'
             '</pre>'))

# Automated EDA via ydata-profiling
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(
        df,
        title='Feedback EDA',
        minimal=True,
        explorative=True
    )
    profile.to_notebook_iframe()
except Exception as e:
    print(f'ydata-profiling skipped ({e}). Showing basic stats instead.')
    print(df['text'].str.len().describe().rename('text_length'))
    if df['label'].notna().any():
        print('\nLabel distribution:')
        print(df['label'].value_counts())

In [ ]:
#@title ⑥ Embedding Stage — Pretrained Transformer (Batched)
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE  = 64

try:
    model = SentenceTransformer(EMBED_MODEL)
    texts = df['text'].tolist()
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    print(f'✅ Embeddings shape: {embeddings.shape}')
    EMBED_OK = True
except Exception as e:
    print(f'⚠️  Embedding failed: {e}. Using TF-IDF fallback.')
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(max_features=512)
    embeddings = vec.fit_transform(df['text']).toarray()
    EMBED_OK = False

print(f'Embedding source: {"sentence-transformers" if EMBED_OK else "TF-IDF fallback"}')

In [ ]:
#@title ⑦ K-Means Clustering + PCA Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

N_CLUSTERS = 7

km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = km.fit_predict(embeddings)
df['cluster'] = labels

sil = silhouette_score(embeddings, labels, sample_size=min(500, len(labels)))
print(f'Silhouette score (k={N_CLUSTERS}): {sil:.4f}')

# PCA 2D projection
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = cm.tab10(np.linspace(0, 1, N_CLUSTERS))

for k in range(N_CLUSTERS):
    mask = labels == k
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=[colors[k]], s=8, alpha=0.6, label=f'C{k}')
axes[0].set_title('PCA 2D Cluster Projection')
axes[0].legend(markerscale=2, fontsize=8)

cluster_sizes = pd.Series(labels).value_counts().sort_index()
axes[1].bar(cluster_sizes.index.astype(str), cluster_sizes.values, color=[colors[i] for i in cluster_sizes.index])
axes[1].set_title('Cluster Sizes')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('cluster_pca.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Clustering complete.')

In [ ]:
#@title ⑧ LLM Summarization — Per-Cluster (OpenAI / Anthropic / Extractive Fallback)
import textwrap

SAMPLE_PER_CLUSTER = 8
cluster_summaries: dict[int, str] = {}

def _extractive_summary(texts: list[str], n: int = 3) -> str:
    """Simple extractive: pick shortest n texts as representative."""
    return ' | '.join(sorted(texts, key=len)[:n])

def _llm_summarize(cluster_id: int, texts: list[str]) -> str:
    prompt = (
        f'Summarize the following {len(texts)} user feedback items from Cluster {cluster_id} '
        f'in 2-3 sentences. Focus on the dominant theme and any safety or ethical signals.\n\n'
        + '\n'.join(f'- {t[:200]}' for t in texts)
    )
    # Try OpenAI
    if OPENAI_API_KEY:
        try:
            import openai
            client = openai.OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=256,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f'OpenAI error: {e}')

    # Try Anthropic
    if ANTHROPIC_API_KEY:
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            resp = client.messages.create(
                model='claude-haiku-4-5-20251001',
                max_tokens=256,
                messages=[{'role': 'user', 'content': prompt}],
            )
            return resp.content[0].text.strip()
        except Exception as e:
            print(f'Anthropic error: {e}')

    # Extractive fallback
    return '[extractive] ' + _extractive_summary(texts)

for cid in range(N_CLUSTERS):
    sample_texts = df[df['cluster'] == cid]['text'].sample(
        min(SAMPLE_PER_CLUSTER, (df['cluster'] == cid).sum()), random_state=42
    ).tolist()
    summary = _llm_summarize(cid, sample_texts)
    cluster_summaries[cid] = summary
    print(f'\n── Cluster {cid} ({(df["cluster"]==cid).sum()} items) ──')
    print(textwrap.fill(summary, width=90))

print('\n✅ Summarization complete.')

In [ ]:
#@title ⑨ Brutalist HTML Report — ARTIFEX-Styled
from IPython.display import display, HTML

rows_html = ''
for cid, summary in cluster_summaries.items():
    size = (df['cluster'] == cid).sum()
    rows_html += f"""
    <tr>
      <td style='padding:8px 12px;border:1px solid #444;font-weight:bold;color:#e8ff00;'>C{cid}</td>
      <td style='padding:8px 12px;border:1px solid #444;'>{size}</td>
      <td style='padding:8px 12px;border:1px solid #444;'>{summary}</td>
    </tr>"""

html = f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Syne+Mono&display=swap');
  .art-report {{ font-family:'Syne Mono',monospace; background:#0a0a0a; color:#e8e8e8;
                 padding:24px; border:2px solid #e8ff00; border-radius:4px; }}
  .art-report h2 {{ color:#e8ff00; letter-spacing:3px; }}
  .art-report table {{ width:100%; border-collapse:collapse; font-size:0.88em; }}
  .art-report th {{ background:#1a1a1a; color:#aaa; padding:8px 12px;
                    border:1px solid #444; text-align:left; }}
</style>
<div class='art-report'>
  <h2>⚡ ARTIFEX · ETHICAL FEEDBACK LOOP · CLUSTER REPORT</h2>
  <p style='color:#888;'>Dataset: {df['source'].iloc[0]} · Rows: {len(df)} · Clusters: {N_CLUSTERS} · Silhouette: {sil:.4f}</p>
  <table>
    <thead><tr>
      <th>CLUSTER</th><th>SIZE</th><th>SUMMARY</th>
    </tr></thead>
    <tbody>{rows_html}</tbody>
  </table>
  <hr style='border-color:#333;margin-top:20px;'>
  <p style='color:#555;font-size:0.8em;'>ARTIFEX Labs · tuesday@artifex.fun · linktr.ee/artifexlabs · zcal.co/tuesday</p>
</div>
"""
display(HTML(html))

In [ ]:
#@title ⑩ Bokeh Dashboard — Cluster Size vs. Sentiment Quality
try:
    from bokeh.plotting import figure, show, output_notebook
    from bokeh.models import ColumnDataSource, HoverTool
    from bokeh.palettes import Category10
    output_notebook()

    # Compute simple sentiment proxy: fraction of positive labels if available
    cluster_stats = []
    for cid in range(N_CLUSTERS):
        sub = df[df['cluster'] == cid]
        size = len(sub)
        if sub['label'].notna().any():
            pos = (sub['label'].astype(str) == '1').mean()
        else:
            # Use PCA centroid distance as quality proxy
            pos = float(np.linalg.norm(km.cluster_centers_[cid]))
        cluster_stats.append({'cluster': f'C{cid}', 'size': size, 'quality': round(pos, 3)})

    source = ColumnDataSource(data={
        'x':       [s['quality'] for s in cluster_stats],
        'y':       [s['size']    for s in cluster_stats],
        'cluster': [s['cluster'] for s in cluster_stats],
        'size':    [s['size']    for s in cluster_stats],
        'quality': [s['quality'] for s in cluster_stats],
    })

    p = figure(title='Cluster Size vs. Sentiment Quality',
               x_axis_label='Quality Score', y_axis_label='Cluster Size',
               width=640, height=400,
               background_fill_color='#0a0a0a', border_fill_color='#111',
               title_location='above')
    p.title.text_color = '#e8ff00'
    p.title.text_font = 'monospace'

    p.scatter('x', 'y', source=source, size=14, color='#e8ff00', alpha=0.8)
    p.add_tools(HoverTool(tooltips=[
        ('Cluster', '@cluster'),
        ('Size', '@size'),
        ('Quality', '@quality'),
    ]))
    show(p)
except Exception as e:
    print(f'Bokeh skipped: {e}')

In [ ]:
#@title ⑪ Reproducibility Tracking — %watermark
%load_ext watermark
%watermark -v -m -p scikit-learn,transformers,datasets,openai,anthropic,pandera,bokeh,matplotlib,numpy,pandas